#  Optimizing a Model with LLM Compressor

In this notebook, you'll:
1. Learn how **post-training quantization** works via full precision & compressed model comparisons
2. Use the `llm-compressor` library to apply a GPTQ recipe that produces a W4A16 quantized model
3. Test and evaluate the quantized model against the original

## What is LLM Compressor?

[llm-compressor](https://github.com/vllm-project/llm-compressor) is the production quantization toolkit from the vLLM project. It takes a trained model and reduces precision in a single pass, no retraining required.

The core API is **`oneshot`**: you give it a model, a calibration dataset (small sample of real inputs used to minimize quantization error), and a recipe describing how to quantize (e.g. GPTQ, W4A16). It produces a smaller model that can be served directly by [vLLM](https://github.com/vllm-project/vllm), an LLM inference engine that you'll use in the next lesson.

```python
oneshot(
    model="model-name",           # HuggingFace model ID
    dataset="dataset-name",       # Calibration dataset
    recipe=recipe,                # Quantization configuration
    output_dir="./output",        # Where to save
    num_calibration_samples=256,  # Samples for calibration
    max_seq_length=4096,          # Sequence length
)
```

The name "oneshot" reflects that this happens in a **single pass** over calibration data, no retraining required.

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MODEL_DIR = "../models/Qwen3-0.6B"
OUTPUT_DIR = "../models/Qwen3-0.6B-W4A16"

print(f"Base model:      {MODEL_DIR}")
print(f"Quantized model: {OUTPUT_DIR}")

Base model:      ../models/Qwen3-0.6B
Quantized model: ../models/Qwen3-0.6B-W4A16


<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>To access the <code>requirements.txt</code> file and the <code>model</code> folder:</b> 1) click the <em>"File"</em> option in the top menu of the notebook, then 2) click <em>"Open"</em>. The <code>requirements.txt</code> file is in the lesson's folder, and the model and output directories are in the <code>model</code> folder.</p>

## Define the Recipe

一个 **Recipe（量化配方）** 告诉 LLM Compressor 应该如何对模型进行量化。它实际上是一个 **Modifier（修改器）列表**：每个 Modifier 都指定了一种算法以及相应的配置参数，这些算法会按照指定方式应用到模型上。

### 可用的 Modifier

**量化类 Modifier：**

| Modifier               | 描述                                                                 |
| :--------------------- | :----------------------------------------------------------------- |
| `GPTQModifier`         | GPTQ 算法：使用校准数据（calibration data）寻找最优的量化值                           |
| `AWQModifier`          | 激活感知权重量化（Activation-Weighted Quantization）：保留重要程度较高的权重，即对模型影响较大的权重 |
| `AutoRoundModifier`    | Intel 的量化算法，通过可学习的舍入（rounding）和截断（clipping）来优化量化效果                 |
| `QuantizationModifier` | 基础的 PTQ（训练后量化）和 QAT（量化感知训练），适用于简单的量化场景                             |

**Transform Modifier（变换类 Modifier）：**

这些 Modifier 主要用于**提高量化后的模型精度**：

| Modifier              | 描述                                                    |
| :-------------------- | :---------------------------------------------------- |
| `SmoothQuantModifier` | 在量化之前对激活值进行平滑处理，通常与 GPTQ 配合使用                         |
| `QuIPModifier`        | 使用 Hadamard 旋转（Hadamard rotations）来减少异常值（outliers）的影响 |
| `SpinQuantModifier`   | 使用 SpinQuant 风格的旋转，使权重分布更加均匀                          |

多个 Modifier 可以进行**链式组合（chaining）**。

例如，可以先应用 `SmoothQuantModifier`，再应用 `GPTQModifier`，从而改善 **W8A8** 量化的精度。

### Quantization Scheme（量化方案）

`scheme` 参数决定了**权重（W）和激活值（A）的位宽**：

| Scheme  | 权重    | 激活值          | 量化层参数/存储减少 | 对模型质量的影响 |
| :------ | :---- | :----------- | :--------- | :------- |
| `W8A16` | 8-bit | 16-bit（FP16） | ~50%       | 极小       |
| `W4A16` | 4-bit | 16-bit（FP16） | ~75%       | 低～中等     |
| `W8A8`  | 8-bit | 8-bit        | ~50%       | 较低       |
| `W4A8`  | 4-bit | 8-bit        | ~75%       | 中等       |

> **注意：**
> 上述减少比例只针对**被量化的层**。Embedding 和 `lm_head` 层保持全精度，因此整个模型最终能够减少多少，取决于这些层占整个模型参数的比例。
>
> 对于较小的模型（约 0.6B 参数），使用 W4A16 时，预计整个模型的总大小可以减少约 **40%～50%**。

### 我们的量化 Recipe：使用 GPTQModifier 进行 W4A16 量化

| 参数        | 值             | 原因                                   |
| :-------- | :------------ | :----------------------------------- |
| `scheme`  | `W4A16`       | 使用 4-bit 权重                          |
| `targets` | `Linear`      | Linear 层包含模型中的大部分参数，因此可以获得最大的压缩收益    |
| `ignore`  | `["lm_head"]` | 输出层负责将隐藏状态映射到词表（vocabulary），因此保持较高精度 |


In [2]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False requires_calibration_data=True index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False block_size=128 dampening_frac=0.01 actorder=static offload_hessians=False


## Quantize the Model

### 为什么需要校准数据集？  
GPTQ 并不是简单地把权重舍入到低精度，而是使用一小部分真实文本来衡量每个权重对模型输出的影响，然后找到能最小化误差的量化值。这正是它比朴素舍入更准确的原因。.

`dataset` 参数指定了用什么文本进行校准。这里你将使用 `WikiText-2` (https://huggingface.co/datasets/mindchain/wikitext2)，一个由维基百科文章组成的标准基准数据集，之后做困惑度评估时也会用它。校准速度很快，只需要几百个样本即可。.

>**Note:** Since quantization can take several minutes and benefits from a GPU, we've already run it ahead of time and provided the quantized model in the `Qwen3-0.6B-W4A16` folder (`OUTPUT_DIR`). This learning environment is memory-constrained, so it might crash if you run the quantization yourself. The `if not os.path.isdir(OUTPUT_DIR)` check below ensures you skip re-running quantization when the folder already exists, so you can move straight to evaluation.


In [3]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=1024,
        num_calibration_samples=128,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Tokenizing:   0%|          | 0/4358 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/36718 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3760 [00:00<?, ? examples/s]

Adding labels:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

Compressing model: 100%|██████████| 196/196 [00:00<00:00, 196.13it/s]


2026-09-08T23:09:50.6127 | _retie_embeddings | INFO - Re-tied input/output embeddings; saving a single shared table.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Dispatching model: 100%|██████████| 427/427 [00:00<00:00, 12941.83it/s]


Quantization complete. Model saved to: ../models/Qwen3-0.6B-W4A16


## Compare Model Sizes

Let's see how much space quantization saves.

In [3]:
def folder_size(path):
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(nbytes):
    if nbytes < 1024**2:
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3:
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig = folder_size(MODEL_DIR)
size_q = folder_size(OUTPUT_DIR)
reduction = (1 - size_q / size_orig) * 100 if size_orig > 0 else 0

print("Model Size Comparison")
print("=" * 45)
print(f"Original (BF16):    {format_size(size_orig)}")
print(f"Quantized (W4A16):  {format_size(size_q)}")
print(f"Reduction:          {reduction:.0f}%")

Model Size Comparison
Original (BF16):    1.41 GB
Quantized (W4A16):  524.4 MB
Reduction:          64%


> **Note**: You might expect a 75% reduction since you went from 16-bit to 4-bit weights (4x smaller), but the actual reduction is 42%. The reason: only the **linear layer weights** are quantized to Int4. The rest of the model (including the LM head and normalization layers) stays at higher precision.
> So the 4x compression applies to the bulk of the parameters (the linear layers, which dominate the model), but the unquantized pieces pull the overall reduction down to ~42%. This ratio improves with larger models, where linear weights make up an even bigger share of total size — a 70B model quantized the same way gets much closer to the theoretical 4x.

## Test Both Models

Smaller files are only useful if the model still produces reasonable output. Let's generate text from both and compare using the Hugging Face [Transformers](https://huggingface.co/docs/transformers/en/index) library, starting with the original model **then** the quantized model.
这两组是 smoke test，不是质量评估。目的只有一个：证明量化后没崩、能跑通同样解码链路。肉眼只能看灾难性坏，看不出质量

In [4]:
prompt = "Machine learning is a branch of"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = base_model.generate(
    **inputs, 
    max_new_tokens=60, 
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Base Model ({MODEL_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Base Model (../models/Qwen3-0.6B)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has gained significant attention in recent years, particularly in the context of the rise of big data and the need for efficient, scalable solutions to complex problems. As the field continues to evolve, the integration of machine learning into various industries is becoming increasingly widespread. However, despite its growing popularity,


In [5]:
import logging
logging.getLogger("llmcompressor").setLevel(logging.WARNING)

quant_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = quant_model.generate(
    **inputs, 
    max_new_tokens=60, 
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Quantized Model ({OUTPUT_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Compressing model: 100%|██████████| 196/196 [00:00<00:00, 11980.58it/s]


Loading weights:   0%|          | 0/702 [00:00<?, ?it/s]

Decompressing model: 100%|██████████| 196/196 [00:02<00:00, 73.93it/s]


Quantized Model (../models/Qwen3-0.6B-W4A16)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has become increasingly popular in the field of machine learning. It is a powerful tool that can be used to solve a variety of problems in the field of machine learning. However, there are some challenges that have been identified in the field of machine learning. For example, the problem of over


## Perplexity Comparison

Side-by-side text gives intuition, but **perplexity** is the standard metric: it measures how well the model predicts text. Lower is better. If quantization has degraded the model, its perplexity will be noticeably higher.

&emsp; 并排看生成只能给直觉，困惑度Perplexity才是标准指标：它衡量模型预测文本的能力，越低越好。如果量化搞坏了模型，困惑度会明显变高。  
$$\boxed{
PPL=\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(x_i|x_{<i})
\right)
}$$

In [6]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

datasets: 5.0.1
huggingface_hub: 1.30.0


In [7]:
from datasets import load_dataset

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

test_data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
print(f"Loaded {len(test_data)} test samples")

'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)' thrown while requesting HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py
Retrying in 1s [Retry 1/5].
'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)' thrown while requesting HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py
Retrying in 2s [Retry 2/5].


Loaded 4358 test samples


In [8]:
quant_ppl = calculate_perplexity(quant_model, tokenizer, test_data)
print(f"Quantized perplexity: {quant_ppl:.2f}")

Quantized perplexity: 37.02


In [9]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)
base_ppl = calculate_perplexity(base_model, tokenizer, test_data)
print(f"Base perplexity: {base_ppl:.2f}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Base perplexity: 32.78


In [10]:
print("Perplexity Comparison")
print("=" * 40)
print(f"Base (BF16):      {base_ppl:.2f}")
print(f"Quantized (W4A16): {quant_ppl:.2f}")
print(f"Difference:       {quant_ppl - base_ppl:+.2f} ({(quant_ppl/base_ppl - 1)*100:+.1f}%)")
print(f"\nA small increase in perplexity is expected — the quantized layers use 4-bit weights.")

Perplexity Comparison
Base (BF16):      32.78
Quantized (W4A16): 37.02
Difference:       +4.24 (+12.9%)

A small increase in perplexity is expected — the quantized layers use 4-bit weights.


## Summary

In this notebook, you:

- Learned how the LLM Compressor **`oneshot`** applies post-training quantization with a GPTQ recipe
- Compared model sizes: W4A16 reduces weights from 16-bit to 4-bit
- Tested both models on the same prompt to verify output quality
- Measured **perplexity** to quantify the accuracy tradeoff

## Resources

- [LLM Compressor GitHub](https://github.com/vllm-project/llm-compressor)
- [LLM Compressor Docs](https://docs.vllm.ai/projects/llm-compressor/en/latest/)